# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My lane: Refresh / Content Opportunity Scoring

Unit of analysis: One row represents one pseudonymized content item for one client on one report date.

Time window: I will use a mid-panel month, such as March 2026, for development and verification. I will avoid using the final _sample month for developing label logic because the final month should be treated as a sealed outcome period.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

For the first feature frame, I will use five features:

GSC impressions — shows how much search visibility the content received.
GSC clicks — shows how many clicks the content received from search.
GSC average position — shows the observed average search position.
GSC sum position — provides an additional measure of search-position performance.
GA4 pageviews — shows observed pageview activity when Analytics data is available.

Available when: These features are based on information observed in the selected data period. For a real prediction, the feature window must end before the decision moment so that future information is not used.

### Label / proxy

The practical decision is whether a content item should be prioritized for review or refresh.

The true business outcome of a refresh is not directly available in this warehouse. Therefore, I will use a measurable proxy for the first analysis rather than claiming to directly predict revenue or the actual value of a refresh.

### Context

I will retain:

client_hash_id
content_hash_id
report_date

These fields identify the observation and its time context. They are context fields and should not automatically be treated as predictive features.

### Excluded

I will exclude:

future-derived information
information derived directly from the target/proxy
client-identifying information
URLs
private queries
any information that could identify a client or domain

These are excluded to reduce leakage and protect the privacy requirements of the dataset.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


Code Cell 1 — Load March

In [ ]:
import pandas as pd

march_rows = []

for row in ds:
    if str(row["report_date"]).startswith("2026-03"):
        march_rows.append(row)

march_df = pd.DataFrame(march_rows)

print("March rows:", len(march_df))
march_df.head()

#### Verification Query 1 — Grain

In [ ]:
print("Total rows:", len(march_df))

unique_client_content_dates = march_df[
    ["client_hash_id", "content_hash_id", "report_date"]
].drop_duplicates()

print(
    "Unique client-content-date combinations:",
    len(unique_client_content_dates)
)

print(
    "Duplicate grain rows:",
    len(march_df) - len(unique_client_content_dates)
)

#### Verification Query 2 — Row count + date span

In [ ]:
print("Row count:", len(march_df))
print("Minimum date:", march_df["report_date"].min())
print("Maximum date:", march_df["report_date"].max())

#### Verification Query 3 — Availability

In [ ]:
available = march_df[
    march_df["gsc_data_available"] == True
]

print("Rows with GSC data available:", len(available))
print("Total March rows:", len(march_df))
print(
    "Percentage with GSC data available:",
    round(len(available) / len(march_df) * 100, 2),
    "%"
)

In [ ]:
print(march_df["gsc_data_available"].value_counts(dropna=False))

### Five Features

In [ ]:
features = march_df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews"
    ]
].copy()

features.head()

In [ ]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_sum_position",
    "ga4_pageviews"
]

feature_frame = march_df[feature_cols].copy()

feature_frame.head()

In [ ]:
print("Five features:")
print(feature_frame.columns.tolist())

print("\nFeature frame shape:", feature_frame.shape)

feature_frame.head()

### Leakage Experiment

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create proxy label
march_df["proxy_declining"] = (
    march_df["gsc_clicks"] < march_df["gsc_impressions"] * 0.01
).astype(int)

# Baseline model without leakage
X = march_df[["gsc_impressions"]].fillna(0)
y = march_df["proxy_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

baseline_score = accuracy_score(y_test, predictions)

print("Baseline score:", round(baseline_score, 4))

### Deliberately add leakage

In [ ]:
# Deliberate leakage: give the model the answer itself
X_leaky = march_df[["gsc_impressions", "proxy_declining"]].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_model = DecisionTreeClassifier(max_depth=4, random_state=42)
leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

leaky_score = accuracy_score(y_test, leaky_predictions)

print("Baseline score:", round(baseline_score, 4))
print("Leaky score:", round(leaky_score, 4))

### Remove leakage

In [ ]:
march_df = march_df.drop(columns=["proxy_declining"])

print("Leaky feature removed.")

### Explanation
Leakage lesson: I deliberately included a feature derived directly from the proxy label. The model was therefore given information about the answer it was supposed to predict. This produced an artificially strong score and does not represent a valid prediction setup. I removed the leaked feature before continuing. A model should only use information that would genuinely be available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can support analysis of observed search and content performance, but it cannot tell me everything about the real-world value of a refresh.

First, the history is unbalanced. Different clients and content items have different amounts of available history, so results may not represent every client or page equally.

Second, some observations may contain Google Search Console data without equivalent Google Analytics information. Therefore, conclusions that depend on Analytics data may apply only to the subset where that information is available.

Third, time windows can overlap. Features and labels must therefore be constructed carefully so that information from the future does not enter the features used at the decision moment.

Fourth, the warehouse contains pseudonymized data. It does not allow me to identify clients, domains, or real search queries, and I will not attempt to re-identify them.

Finally, the model can provide decision support and ranking signals, but it cannot by itself prove that refreshing a page will increase revenue, traffic, or conversions. Those outcomes would require appropriate future observations or experiments.

**Named limitation:** Unbalanced history.

Because different clients and content items have different history depths, the observations available in March 2026 may not represent the same amount of historical information for every page. Therefore, any findings should be treated as directional decision-support rather than universal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.